In [ ]:
import os
from PIL import Image, ImageOps, ImageDraw, ImageFont

# =====================================================
# Settings
# =====================================================
root_dir = r"D:\sc_modelling-amir-2025"

species_list = ["Mouse", "Human"]

model_order = [
    "Hybrid_old_BE_fit",
    "Hybrid_BE_fit",
    "Hybrid_new3_fit",
    "Hybrid_old_SC_fit",
    "Hybrid_SC_fit",
]

# Layout (3 plots on top, 2 on bottom)
row_layout = [
    ["Hybrid_old_BE_fit", "Hybrid_BE_fit", "Hybrid_new3_fit"],
    ["Hybrid_old_SC_fit", "Hybrid_SC_fit"]
]

# Names to show above plots
display_names = {
    "Hybrid_old_BE_fit": "Old BE",
    "Hybrid_BE_fit": "New BE",
    "Hybrid_new3_fit": "Hybrid",
    "Hybrid_old_SC_fit": "Old SC",
    "Hybrid_SC_fit": "New SC"
}

subfolders = [
    "summary_outputs_per_participant",
    "summary_outputs_per_participant2",
    "matrix_plots_per_participant",
    "summary_outputs"
]

output_root = os.path.join(
    root_dir,
    "combined_model_comparison_figures"
)
os.makedirs(output_root, exist_ok=True)

# =====================================================
# Appearance
# =====================================================
TITLE_HEIGHT = 90
FONT_SIZE = 42

GAP = 30
MARGIN = 40


# =====================================================
# Utilities
# =====================================================
def list_pngs(folder):
    if not os.path.isdir(folder):
        return set()

    return {
        f for f in os.listdir(folder)
        if f.lower().endswith(".png")
    }


def make_title_bar(text, width, height=TITLE_HEIGHT):

    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("arial.ttf", FONT_SIZE)
    except:
        font = ImageFont.load_default()

    bbox = draw.textbbox((0, 0), text, font=font)

    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    x = (width - text_w) // 2
    y = (height - text_h) // 2

    draw.text((x, y), text, fill="black", font=font)

    return img


def resize_keep_aspect(img, target_w, target_h):

    img = ImageOps.contain(img, (target_w, target_h))

    canvas = Image.new("RGB", (target_w, target_h), "white")

    x = (target_w - img.width) // 2
    y = (target_h - img.height) // 2

    canvas.paste(img, (x, y))

    return canvas


# =====================================================
# Combine one figure
# =====================================================
def combine_one_figure(species, subfolder, filename, save_path):

    loaded = {}

    # Load images
    for model in model_order:

        path = os.path.join(
            root_dir,
            model,
            species,
            "results",
            subfolder,
            filename
        )

        if os.path.isfile(path):
            loaded[model] = Image.open(path).convert("RGB")

    if len(loaded) == 0:
        return

    # Determine common size
    max_w = max(img.width for img in loaded.values())
    max_h = max(img.height for img in loaded.values())

    cell_w = max_w
    cell_h = max_h + TITLE_HEIGHT

    canvas_w = (
        3 * cell_w
        + 2 * GAP
        + 2 * MARGIN
    )

    canvas_h = (
        2 * cell_h
        + GAP
        + 2 * MARGIN
    )

    canvas = Image.new(
        "RGB",
        (canvas_w, canvas_h),
        "white"
    )

    # -------------------------------------------------
    # Draw rows
    # -------------------------------------------------
    for row_idx, row_models in enumerate(row_layout):

        n_cols = len(row_models)

        row_width = (
            n_cols * cell_w
            + (n_cols - 1) * GAP
        )

        # Center each row
        start_x = (canvas_w - row_width) // 2

        y = MARGIN + row_idx * (cell_h + GAP)

        for col_idx, model in enumerate(row_models):

            x = start_x + col_idx * (cell_w + GAP)

            # Title
            title_bar = make_title_bar(
                display_names[model],
                cell_w
            )

            canvas.paste(title_bar, (x, y))

            # Plot
            if model in loaded:

                plot_img = resize_keep_aspect(
                    loaded[model],
                    cell_w,
                    max_h
                )

            else:

                plot_img = Image.new(
                    "RGB",
                    (cell_w, max_h),
                    "white"
                )

                draw = ImageDraw.Draw(plot_img)

                try:
                    font = ImageFont.truetype(
                        "arial.ttf",
                        30
                    )
                except:
                    font = ImageFont.load_default()

                draw.text(
                    (50, 50),
                    "Missing",
                    fill="red",
                    font=font
                )

            canvas.paste(
                plot_img,
                (x, y + TITLE_HEIGHT)
            )

    os.makedirs(
        os.path.dirname(save_path),
        exist_ok=True
    )

    canvas.save(
        save_path,
        dpi=(300, 300)
    )


# =====================================================
# Main loop
# =====================================================
for species in species_list:

    print(f"\nProcessing {species}")

    for subfolder in subfolders:

        print(f"  {subfolder}")

        all_pngs = set()

        # Collect all png names from all model folders
        for model in model_order:

            folder = os.path.join(
                root_dir,
                model,
                species,
                "results",
                subfolder
            )

            all_pngs |= list_pngs(folder)

        if len(all_pngs) == 0:
            continue

        out_dir = os.path.join(
            output_root,
            species,
            subfolder
        )

        os.makedirs(out_dir, exist_ok=True)

        for filename in sorted(all_pngs):

            stem = os.path.splitext(filename)[0]

            save_path = os.path.join(
                out_dir,
                stem + "_combined.png"
            )

            combine_one_figure(
                species,
                subfolder,
                filename,
                save_path
            )

            print("    Saved:", stem)


Processing Mouse
  summary_outputs_per_participant
    Saved: QP0100_parameter_profile
    Saved: QP0101_parameter_profile
    Saved: QP0103_parameter_profile
    Saved: QP0121_parameter_profile
    Saved: QP062_parameter_profile
    Saved: QP063_parameter_profile
    Saved: QP070_parameter_profile
    Saved: QP071_parameter_profile
  summary_outputs_per_participant2
    Saved: QP0100_parameter_profile
    Saved: QP0101_parameter_profile
    Saved: QP0103_parameter_profile
    Saved: QP0121_parameter_profile
    Saved: QP062_parameter_profile
    Saved: QP063_parameter_profile
    Saved: QP070_parameter_profile
    Saved: QP071_parameter_profile
  matrix_plots_per_participant
    Saved: MEAN_update_matrices
    Saved: QP0100_update_matrices
    Saved: QP0101_update_matrices
    Saved: QP0103_update_matrices
    Saved: QP0121_update_matrices
    Saved: QP062_update_matrices
    Saved: QP063_update_matrices
    Saved: QP070_update_matrices
    Saved: QP071_update_matrices
  summary_outp

: 